# bubbleT — Q-learning

36 states × 6 actions. Training uses CPU.

For state $s_t=(m_t,d_t,b_t)$, the table row is $12m_t+2(d_t+1)+b_t$.

$r_t$ is the reward we define for each transition.
$$\delta_t = r_t + \gamma(1-D_t)\max_a Q(s_{t+1},a)-Q(s_t,a_t)$$
$$Q(s_t,a_t) \leftarrow Q(s_t,a_t)+\alpha\delta_t$$
$$\mathrm{TD\ MSE}=\frac{1}{N}\sum_{i=1}^{N}\delta_i^2$$

$D_t=1$ at death or episode end. 
Each epoch visits every training transition once, in shuffled order. Validation holds out whole episodes.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

PROJECT = Path('/export/home/ra26xel/bb_rl/bomberman_rl-master')
sys.path.insert(0, str(PROJECT))
from agent_code.bubbleT.q_table import ACTIONS, CHECKPOINT
from agent_code.bubbleT.trainer import Trainer, load_replay


In [ ]:
SCRATCH = Path('/export/scratch/ra26xel/bomberman')
stage2 = load_replay([SCRATCH / 'triple_scraping_160k/stage_2'])
stage3 = load_replay([
    SCRATCH / 'triple_scraping_160k/stage_3',
    SCRATCH / 'triple_scraping_stage3_extra200k/stage_3',
    SCRATCH / 'triple_scraping_stage3_extra1640k/stage_3',
])


Resume restores the saved settings. Epoch counts are **totals**, so increase stage 3 to continue.
For a fresh run, archive `bubbleT.npz` and set `RESUME=False`. Set `guidance_weight=0` for score-only rewards.


In [ ]:
RESUME = True
STAGE2_EPOCHS = 10
STAGE3_EPOCHS = 20

if RESUME:
    trainer = Trainer.resume(CHECKPOINT)
else:
    if CHECKPOINT.exists():
        raise FileExistsError('Archive the existing checkpoint before starting fresh.')
    trainer = Trainer(alpha=.05, gamma=.92, guidance_weight=1., seed=42)
print(trainer.config)


In [ ]:
def plot_losses(history):
    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(8, 3))
    epochs = [row['epoch'] for row in history]
    ax.plot(epochs, [row['train_loss'] for row in history], label='Training')
    ax.plot(epochs, [row['validation_loss'] for row in history], label='Validation')
    for previous, current in zip(history, history[1:]):
        if previous['stage'] != current['stage']:
            ax.axvline(current['epoch'] - .5, color='gray', linestyle='--')
    ax.set(xlabel='Epoch', ylabel='TD MSE', title=history[-1]['stage'])
    ax.legend()
    display(fig)
    plt.close(fig)
    print(history[-1])
    print('Checkpoint:', CHECKPOINT)

stage2_done = sum(row['stage'] == 'stage_2' for row in trainer.history)
trainer.fit(stage2, max(0, STAGE2_EPOCHS - stage2_done), 'stage_2', checkpoint=CHECKPOINT, on_epoch=plot_losses)

stage3_done = sum(row['stage'] == 'stage_3' for row in trainer.history)
trainer.fit(stage3, max(0, STAGE3_EPOCHS - stage3_done), 'stage_3', checkpoint=CHECKPOINT, on_epoch=plot_losses)

if trainer.history:
    plot_losses(trainer.history)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 8))
im = ax.imshow(trainer.q, aspect='auto')
ax.set_xticks(range(6), ACTIONS, rotation=45)
ax.set(xlabel='Action', ylabel='State index', title='Q-table')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


Saved after every epoch to `bubbleT.npz`. Training TD errors are measured during updates; validation errors at epoch end. Evaluate **points/game** to compare policies.
